Parsing Raw Log Data

In [ ]:
import re
import pandas as pd
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig

LOG_FILE = "HDFS.log"  # full dataset (~11M lines); use "HDFS_2k.log" for the 2k sample

with open(LOG_FILE, "r") as f:
    raw_lines = f.readlines()

print(f"Loaded {len(raw_lines):,} log lines from {LOG_FILE}")

# TemplateMinerConfig.__init__ already sets defaults; just override what we need.
# Passing config= suppresses the "config file not found" warning.
config = TemplateMinerConfig()
config.drain_sim_th = 0.5     # higher threshold → fewer, coarser templates
config.drain_depth = 4
config.drain_max_children = 100
miner = TemplateMiner(config=config)

parsed = []

for line in raw_lines:
    # Skip lines with no block_id (avoids a spurious None session)
    match = re.search(r'(blk_[-\d]+)', line)
    if not match:
        continue
    block_id = match.group(1)

    # Strip the HDFS prefix (Date Time PID Level) before feeding to Drain3.
    # Raw format: "081109 203615 148 INFO dfs.DataNode$...: message"
    # Without this, timestamps make every line look unique and inflate the template count.
    parts = line.strip().split(None, 4)
    message = parts[4] if len(parts) > 4 else line.strip()

    result = miner.add_log_message(message)
    log_key = result["template_mined"]

    parsed.append({"block_id": block_id, "log_key": log_key})

df = pd.DataFrame(parsed)
print(f"Unique log keys (templates): {df['log_key'].nunique()}")  # expect ~29 for full HDFS
print(df.head())


Air Passengers Dataset
Yahoo Finance Stock Data

Loaded 11,175,629 log lines from HDFS.log
Unique log keys (templates): 115
                   block_id                                            log_key
0  blk_-1608999687919862906  dfs.DataNode$DataXceiver: Receiving block blk_...
1  blk_-1608999687919862906  dfs.FSNamesystem: BLOCK* NameSystem.allocateBl...
2  blk_-1608999687919862906  dfs.DataNode$DataXceiver: Receiving block blk_...
3  blk_-1608999687919862906  dfs.DataNode$DataXceiver: Receiving block blk_...
4  blk_-1608999687919862906  dfs.DataNode$PacketResponder: PacketResponder ...


Group by block_id into sessions

In [3]:
# Load anomaly labels
labels = pd.read_csv("anomaly_label.csv")
# BlockId   Label
# blk_...   Normal / Anomaly

# Encode all unique log keys as integers
unique_keys = df["log_key"].unique().tolist()
key2id = {k: i for i, k in enumerate(unique_keys)}
df["key_id"] = df["log_key"].map(key2id)
num_keys = len(unique_keys)  # should be ~29 for HDFS

# Group into sessions
sessions = df.groupby("block_id")["key_id"].apply(list).reset_index()
sessions.columns = ["block_id", "sequence"]

# Merge with labels
sessions = sessions.merge(labels, left_on="block_id", right_on="BlockId")
sessions["label"] = (sessions["Label"] == "Anomaly").astype(int)

print(f"Total sessions: {len(sessions)}")
print(f"Normal: {(sessions.label==0).sum()}, Anomaly: {(sessions.label==1).sum()}")
# Normal: 558366, Anomaly: 16838

Total sessions: 575061
Normal: 558223, Anomaly: 16838


In [4]:
# Use only normal sessions for training (mimicking the paper: ~4855 sessions)
normal_sessions = sessions[sessions.label == 0]
train_sessions  = normal_sessions.sample(frac=0.8, random_state=42)

# Everything else is test data (normal + anomalous)
test_sessions = sessions.drop(train_sessions.index)

print(f"Train: {len(train_sessions)} normal sessions")
print(f"Test:  {len(test_sessions)} sessions ({test_sessions.label.sum()} anomalous)")

Train: 446578 normal sessions
Test:  128483 sessions (16838 anomalous)


Build sliding window sequences

In [5]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

WINDOW_SIZE = 10  # h = 10 (history window)

def make_windows(sequences):
    X, Y = [], []
    for seq in sequences:
        for i in range(len(seq) - WINDOW_SIZE):
            X.append(seq[i : i + WINDOW_SIZE])   # input: 10 log keys
            Y.append(seq[i + WINDOW_SIZE])         # label: next log key
    return np.array(X), np.array(Y)

X_train, Y_train = make_windows(train_sessions["sequence"].tolist())
print(f"Training samples: {len(X_train)}")

if len(X_train) == 0:
    avg_len = train_sessions["sequence"].apply(len).mean()
    raise ValueError(
        f"No training windows generated. Average session length is {avg_len:.1f} events, "
        f"but WINDOW_SIZE={WINDOW_SIZE} requires sessions longer than {WINDOW_SIZE}.\n"
        f"The HDFS_2k.log dataset has ~1 event/session — switch to the full HDFS dataset "
        f"(~11M lines, ~575k sessions) for DeepLog to work as described in the paper."
    )

class LogDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.LongTensor(X)
        self.Y = torch.LongTensor(Y)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.Y[i]

train_loader = DataLoader(LogDataset(X_train, Y_train),
                          batch_size=2048, shuffle=True)

Training samples: 4244008


Build and train the LSTM model

In [6]:
import torch.nn as nn
import os

class DeepLog(nn.Module):
        def __init__(self, num_keys, hidden_size=64, num_layers=2):
            super().__init__()
            self.embed    = nn.Embedding(num_keys, hidden_size)
            self.lstm     = nn.LSTM(hidden_size, hidden_size,
                                num_layers=num_layers, batch_first=True)
            self.fc       = nn.Linear(hidden_size, num_keys)
    
        def forward(self, x):
            x   = self.embed(x)          # (batch, window, hidden)
            out, _ = self.lstm(x)        # (batch, window, hidden)
            out = self.fc(out[:, -1, :]) # take last timestep → (batch, num_keys)
            return out

# ── Training ────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if os.path.exists("deeplog_hdfs.pt"):
      model = DeepLog(num_keys=num_keys, hidden_size=64, num_layers=2).to(device)
      model.load_state_dict(torch.load("deeplog_hdfs.pt", map_location=device))
      print("Loaded saved model from deeplog_hdfs.pt")
else:
    model  = DeepLog(num_keys=num_keys, hidden_size=64, num_layers=2).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    NUM_EPOCHS = 5
    for epoch in range(NUM_EPOCHS):
        model.train()
        total_loss = 0
        for X_batch, Y_batch in train_loader:
            X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)
            optimizer.zero_grad()
            output = model(X_batch)              # (batch, num_keys)
            loss   = criterion(output, Y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            print(f"Epoch {epoch+1}/{NUM_EPOCHS}  Loss: {total_loss/len(train_loader):.4f}")

    torch.save(model.state_dict(), "deeplog_hdfs.pt")

Loaded saved model from deeplog_hdfs.pt


Run anomaly detection

In [7]:
G = 9  # top-g candidates considered normal

def predict_session(model, sequence, window_size=10, g=36):
    """Returns True if session is anomalous."""
    model.eval()
    with torch.no_grad():
        for i in range(len(sequence) - window_size):
            window     = sequence[i : i + window_size]
            actual_key = sequence[i + window_size]
            
            x      = torch.LongTensor([window]).to(device)
            output = model(x)                        # (1, num_keys)
            probs  = torch.softmax(output, dim=1)
            
            # Top-g predicted keys
            top_g = torch.topk(probs, g).indices[0].tolist()
            
            if actual_key not in top_g:
                return True  # anomaly detected at this step
    return False  # all steps were normal

# ── Run on test set ──────────────────────────────────────────────────────
results = []
for _, row in test_sessions.iterrows():
    pred = predict_session(model, row["sequence"])
    results.append({
        "block_id":   row["block_id"],
        "true_label": row["label"],
        "pred_label": int(pred)
    })

results_df = pd.DataFrame(results)

Evaluate results

In [8]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

y_true = results_df["true_label"]
y_pred = results_df["pred_label"]

precision = precision_score(y_true, y_pred)
recall    = recall_score(y_true, y_pred)
f1        = f1_score(y_true, y_pred)
cm        = confusion_matrix(y_true, y_pred)

print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F-measure : {f1:.4f}")
print(f"\nConfusion Matrix:\n{cm}")
# Expected (matching paper):
# Precision: ~0.95   Recall: ~1.00   F-measure: ~0.96

Precision : 1.0000
Recall    : 0.0301
F-measure : 0.0585

Confusion Matrix:
[[111645      0]
 [ 16331    507]]
